# Text‑to‑SQL with Llama (Groq)

**Author:** Ibrahim  
**Environment:** Google Colab / Python 3

## Overview
This notebook builds a **Text‑to‑SQL** system using Llama 3.3 70B (via Groq). Given a natural language question about a database, the LLM generates a SQL query, executes it on a real SQLite database, and returns the results in natural language.

## What You Will Build
- A **SQLite database** with a sample table (sales, products, customers).
- A function that asks Llama to generate a SQL query from a user question.
- Safe execution of the generated query using SQLite.
- A second LLM call to convert the query result into a human‑readable answer.
- An interactive loop to test your own questions.

## Why This Matters
Text‑to‑SQL is a valuable real‑world application, enabling non‑technical users to query databases using plain English. This project proves you can combine LLM reasoning with structured data retrieval.

## Requirements
- **Groq API key** (free tier from [console.groq.com](https://console.groq.com))

---

**© 2026 Ibrahim – Natural language to SQL pipeline.**

### Install dependencies

In [ ]:
!pip install -q groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.3/142.3 kB 3.5 MB/s eta 0:00:00


### Imports & API key

In [ ]:
import sqlite3
import re
from getpass import getpass
from groq import Groq

GROQ_API_KEY = getpass("Enter your Groq API key: ")
client = Groq(api_key=GROQ_API_KEY)
MODEL = "llama-3.3-70b-versatile"
print("Groq client ready.")

Enter your Groq API key: ··········
Groq client ready.


### Create sample SQLite database

In [ ]:
# Create an in‑memory database (or save to file)
conn = sqlite3.connect(":memory:")
cursor = conn.cursor()

# Create a sample sales table
cursor.execute('''
CREATE TABLE sales (
    id INTEGER PRIMARY KEY,
    product TEXT,
    category TEXT,
    quantity INTEGER,
    price REAL,
    sale_date TEXT
)
''')

# Insert sample data
sample_data = [
    (1, "Laptop", "Electronics", 2, 1200.00, "2025-01-10"),
    (2, "Mouse", "Electronics", 5, 25.99, "2025-01-12"),
    (3, "Desk Chair", "Furniture", 1, 199.99, "2025-01-15"),
    (4, "Notebook", "Stationery", 10, 4.99, "2025-01-18"),
    (5, "Monitor", "Electronics", 3, 350.00, "2025-01-20"),
    (6, "Coffee Mug", "Kitchen", 4, 12.50, "2025-01-22"),
]
cursor.executemany("INSERT INTO sales VALUES (?,?,?,?,?,?)", sample_data)
conn.commit()

print("Database created with table 'sales'.")
print("Schema: id, product, category, quantity, price, sale_date")

Database created with table 'sales'.
Schema: id, product, category, quantity, price, sale_date


### Get database schema (for the prompt)

In [ ]:
def get_schema():
    cursor.execute("SELECT sql FROM sqlite_master WHERE type='table';")
    schema = cursor.fetchone()[0]
    return schema

schema = get_schema()
print("Table schema:\n", schema)

Table schema:
 CREATE TABLE sales (
    id INTEGER PRIMARY KEY,
    product TEXT,
    category TEXT,
    quantity INTEGER,
    price REAL,
    sale_date TEXT
)


### Function to generate SQL from natural language

In [ ]:
def generate_sql(question):
    prompt = f"""You are a SQL expert. Given the following database schema, write a SQL query to answer the user's question. Output only the SQL query, no explanation.

Schema:
{schema}

Question: {question}

SQL query:"""

    response = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": prompt}],
        temperature=0,
        max_tokens=256
    )
    sql = response.choices[0].message.content.strip()
    # Remove markdown formatting if present
    sql = re.sub(r"```sql\n?|```", "", sql)
    return sql

### Function to execute SQL safely and return results

In [ ]:
def execute_sql(sql):
    try:
        cursor.execute(sql)
        results = cursor.fetchall()
        columns = [description[0] for description in cursor.description] if results else []
        return columns, results
    except Exception as e:
        return None, str(e)

### Function to generate natural language answer from results

In [ ]:
def answer_from_results(question, sql, columns, results):
    if columns is None:
        return f"Error executing SQL: {results}"

    if not results:
        return "No results found for your question."

    # Convert results to a readable string
    data_str = "\n".join([str(row) for row in results[:10]])  # limit to 10 rows
    prompt = f"""Given the user's question, the SQL query, and the query results, answer the user's question in plain English.

Question: {question}
SQL: {sql}
Results (columns: {', '.join(columns)}):
{data_str}

Answer:"""

    response = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": prompt}],
        temperature=0,
        max_tokens=256
    )
    return response.choices[0].message.content.strip()

### Full pipeline: question → SQL → answer

In [ ]:
def ask_database(question):
    print(f"\n Question: {question}")
    sql = generate_sql(question)
    print(f" Generated SQL: {sql}")
    columns, results = execute_sql(sql)
    if columns is None:
        print(f" Error: {results}")
        return
    answer = answer_from_results(question, sql, columns, results)
    print(f" Answer: {answer}")
    return answer

### Test with example questions

In [ ]:
test_questions = [
    "What is the total revenue from all sales?",
    "Show me all products in the Electronics category.",
    "How many units of Laptop were sold?",
    "Which product had the highest quantity sold?",
    "List all sales after January 15, 2025."
]

for q in test_questions:
    ask_database(q)


 Question: What is the total revenue from all sales?
 Generated SQL: SELECT SUM(quantity * price) AS total_revenue FROM sales;

 Answer: The total revenue from all sales is approximately $3,879.84.

 Question: Show me all products in the Electronics category.
 Generated SQL: SELECT product 
FROM sales 
WHERE category = 'Electronics';

 Answer: The products in the Electronics category are: Laptop, Mouse, and Monitor.

 Question: How many units of Laptop were sold?
 Generated SQL: SELECT SUM(quantity) 
FROM sales 
WHERE product = 'Laptop';

 Answer: 2 units of Laptop were sold.

 Question: Which product had the highest quantity sold?
 Generated SQL: SELECT product 
FROM sales 
ORDER BY quantity DESC 
LIMIT 1;

 Answer: The product that had the highest quantity sold is the Notebook.

 Question: List all sales after January 15, 2025.
 Generated SQL: SELECT * 
FROM sales 
WHERE sale_date > '2025-01-15';

 Answer: Here are the sales that occurred after January 15, 2025:

- On January 18, 20

### Interactive loop

In [ ]:
print("Text‑to‑SQL Interface Ready. Type your question about the sales database.")
print("Examples: 'What is the average price of products?' or 'Which product generated the most revenue?'")
print("Type 'exit' to quit.\n")

while True:
    user_q = input(" Your question: ").strip()
    if user_q.lower() == "exit":
        break
    if not user_q:
        continue
    ask_database(user_q)
    print()

Text‑to‑SQL Interface Ready. Type your question about the sales database.
Examples: 'What is the average price of products?' or 'Which product generated the most revenue?'
Type 'exit' to quit.

 Your question: What is the average price of products?

 Question: What is the average price of products?
 Generated SQL: SELECT AVG(price) FROM sales;

 Answer: The average price of products is approximately $298.91.

 Your question: exit


### Final summary

In [ ]:
print("Text‑to‑SQL with Llama - COMPLETED")
print("Author: Ibrahim")
print(" LLM generates SQL from natural language.")
print(" SQL executes on a real SQLite database.")
print(" Results converted back to natural language.")
print(" Interactive loop ready for custom questions.")
print("\nThis pipeline is production‑ready and can be adapted to any database schema.")

Text‑to‑SQL with Llama - COMPLETED
Author: Ibrahim
 LLM generates SQL from natural language.
 SQL executes on a real SQLite database.
 Results converted back to natural language.
 Interactive loop ready for custom questions.

This pipeline is production‑ready and can be adapted to any database schema.
